# 🕴️ The Controller — the bouncer at the door

In `06_blockchain_from_zero.ipynb` and `07_chainmcp_the_signing_adapter.ipynb` the chain
became real: ticket #7 is a genuine ERC-721 that a genuine contract minted when Ada
paid. This chapter meets the component that decides what that ticket is *worth* in the
physical world: the **controller** — the bouncer standing between an on-chain promise
and a configured router. A real ticket gets you to the door; the bouncer still runs a
checklist before anything turns on.

**What you'll be able to do after this notebook**

- Walk the real predicate check by check and trigger *every* deny path by flipping one fact
- Prove rule 4 live (the judgment imports no I/O) with an AST scan of `domain.py`
- Run the challenge–response dance with a real key, then defeat your own replay attack
- Pin the translator's output against the repo's eye-reviewed golden file
- Stand the whole controller up on cardboard and drive its HTTP API in-process — including tearing a session down twice (rule 8)

**You need:** notebooks 00–07. No chain, no router, no LLM — after two chapters of real
Anvil, we are deliberately back to pure in-process Python, running on 05's fakes. That
the bouncer works *identically* with the chain unplugged is not a convenience; it is the
lesson.

**Estimated time:** 60–90 minutes.

> The compact one-pass tour of the same code is
> [`e2e/notebooks/controller_explore.ipynb`](../controller_explore.ipynb) — replay it
> after this chapter and every cell in it will feel obvious.

## 1. Necessary, but not sufficient — and why the bouncer is never an LLM

Ticket #7 is cryptographic fact: Ada paid, the contract minted, `ownerOf(7)` says Ada.
And yet a real ticket alone must not open the valve. Is the person *asking* actually
Ada? Is it 14:02 — inside her 14:00→16:00 window — or 13:00, or 16:00:01? Did Bell
revoke the ticket an hour ago? Is it a service type this controller even knows how to
provision? Is ticket #7 *already* running a session? A real ticket answers none of
those. The bouncer's checklist does.

Who should run the checklist? This repo's **rule 1** (its most important line): *the
controller and its predicate are never an LLM and never call one.* Three reasons, in
rising order of importance:

- **Auditability** — every "no" must cite a named check. "The model felt uneasy" is not
  a denial reason anyone can act on.
- **Replayability** — same facts in, same answer out, today and in five years. A test
  suite, a dispute process, or a thesis evaluation can re-run the exact decision.
- **The thesis itself** — the claim under evaluation is that *payment → entitlement →
  enforcement* can be **mechanical**. Put a probabilistic model at the enforcement
  boundary and the claim quietly dissolves.

LLM judgment exists in exactly two places in this system — the consumer's accept/reject
and the provider's quote/decline — and both live in `10_agents_the_brains.ipynb`.
Everything in *this* chapter is deterministic code you can read to the bottom. First,
the cast in one cell.

In [ ]:
import inspect
import time
from datetime import datetime, timezone

from a2a_interfaces import ErrorCode, SessionState
from a2a_interfaces import fixtures as fx
from controller.domain import (
    KNOWN_SERVICE_TYPES,
    TRANSITIONS,
    IllegalTransition,
    predicate,
    transition,
)


def hhmm(ts: int) -> str:
    """Unix seconds -> the story's HH:MM (UTC), for readable narration."""
    return datetime.fromtimestamp(ts, tz=timezone.utc).strftime("%H:%M")


VIEW = fx.CANONICAL_ENTITLEMENT_VIEW      # ticket #7 exactly as the controller reads it (04)
IN_WINDOW = fx.WINDOW.start + 120         # 14:02 — two minutes into Ada's window
print("ticket #7 →", hhmm(VIEW.start_time), "→", hhmm(VIEW.end_time),
      "| service_type:", VIEW.service_type, "| revoked:", VIEW.revoked)
print("known service types this controller can translate:", KNOWN_SERVICE_TYPES)

One continuity check before we start. In `05_the_walking_skeleton.ipynb` you already ran
a predicate — imported from the *stub* controller. That was never a throwaway copy: M4.1
lifted it into `controller/domain.py` and the stub now imports it back. The name
`_domain_predicate` starts with `_` — internal, we peek only to learn:

In [ ]:
from controller import domain
from e2e.skeleton import stub_controller

assert stub_controller._domain_predicate is predicate
print("05's predicate and controller.domain.predicate → the SAME function object ✓")
print("(M4.1 lifted it; there is exactly one bouncer in the codebase)")

## 2. The predicate: a pure function where even *time* is an argument

What breaks if the bouncer reads the clock, queries the chain, or pings the router
itself? You could never test a denial without standing up that infrastructure, never
replay a decision (the clock has moved on), and never *prove* what the decision depended
on. So `domain.py` holds the **pure-function ideal**: same inputs → same output, no I/O,
no hidden state — and, crucially, **no clock reads**. `now` arrives as an argument,
fetched by the caller from the *chain's* clock (ADR-004: chain time is the only clock —
you watched `FakeClock` stand in for it in 05). Likewise `owner` is `ownerOf(id)`
fetched by someone else, and `view` is the ticket as read from the chain.

Read the whole judgment — it fits on one screen:

In [ ]:
print(inspect.getsource(predicate))

Six checks, one contract order (**who → not-yet → expired → revoked → scope →
conflict**), and the return convention you met in `01_python_toolbox.ipynb` as a union
type: `ErrorCode | None`, where **`None` means "come in"** and the first failing check
names the refusal. Two checks deserve a second look:

- `now >= view.end_time` — the window is **half-open**: `[start, end)`. 15:59:59 is
  service; 16:00:00 sharp is not.
- `view.service_type not in known_service_types` — "in scope" means *this controller
  has a translator for it* (§7). Admitting a type it can't translate would pass the
  door and crash mid-provision; `E_SCOPE` is the honest early answer.

Run the accept case: 14:02, Ada asks with her own ticket, no session active.

In [ ]:
verdict = predicate(VIEW, owner=fx.ADA, requester=fx.ADA, now=IN_WINDOW, active_ids=set())
print("14:02, Ada asks →", verdict, " ← None means come in ✓")
assert verdict is None

**✏️ Your turn 2.1 — the last second of service (predict, then check)**

The window ends at 16:00:00. Write your prediction for 15:59:59 and for 16:00:00 sharp
as strings *before* running the cell, then un-comment the assert. Success: both
predictions match, and you can say which single character of `domain.py` makes it so.

In [ ]:
# TODO: write your predictions BEFORE running — "None" or an ErrorCode name
prediction_at_155959 = "?"
prediction_at_160000 = "?"

last_second = predicate(VIEW, fx.ADA, fx.ADA, now=fx.WINDOW.end - 1, active_ids=set())
at_four = predicate(VIEW, fx.ADA, fx.ADA, now=fx.WINDOW.end, active_ids=set())
print("15:59:59 →", last_second)
print("16:00:00 →", at_four)

# un-comment once your predictions match:
# assert last_second is None and at_four == ErrorCode.E_EXPIRED

<details><summary>✅ Solution 2.1 — peek only after trying</summary>

```python
prediction_at_155959 = "None"
prediction_at_160000 = "E_EXPIRED"
assert last_second is None and at_four == ErrorCode.E_EXPIRED
```

The check is `now >= view.end_time` — half-open `[start, end)`; were it `>`, every
ticket would quietly include one free second past what Ada paid for.

</details>

## 3. Every deny path, one flipped fact at a time

Now break each check on purpose — flip exactly **one** fact per case and watch the named
refusal. The "edited" tickets use `model_copy` because `EntitlementView` is frozen (02:
you never mutate a shape, you derive a new one). Each denial is an `ErrorCode` from the
shared enum in `a2a_interfaces` — the same strings the HTTP API returns in §11 and the
agents read in chapter 10.

In [ ]:
cases = {
    "Bell asks instead of Ada": (dict(view=VIEW, requester=fx.BELL, now=IN_WINDOW), ErrorCode.E_NOT_OWNER),
    "Ada asks at 13:00":        (dict(view=VIEW, requester=fx.ADA, now=fx.WINDOW.start - 3600), ErrorCode.E_NOT_STARTED),
    "Ada asks at 16:00 sharp":  (dict(view=VIEW, requester=fx.ADA, now=fx.WINDOW.end), ErrorCode.E_EXPIRED),
    "Bell revoked the ticket":  (dict(view=VIEW.model_copy(update={"revoked": True}), requester=fx.ADA, now=IN_WINDOW), ErrorCode.E_REVOKED),
    "serviceType nobody knows": (dict(view=VIEW.model_copy(update={"service_type": 9}), requester=fx.ADA, now=IN_WINDOW), ErrorCode.E_SCOPE),
}
for story, (kw, expected) in cases.items():
    got = predicate(owner=fx.ADA, active_ids=set(), **kw)
    print(f"{story:26} → {got.value:14} {'✓' if got == expected else '✗'}")
    assert got == expected

conflict = predicate(VIEW, fx.ADA, fx.ADA, IN_WINDOW, active_ids={7})
print(f"{'#7 already has a session':26} → {conflict.value:14} ✓")
assert conflict == ErrorCode.E_CONFLICT

**The check order is a documented contract, not an accident.** A request that fails
*everything at once* still reports the FIRST check — so error strings are stable, tests
can pin them, and an agent reading `E_NOT_OWNER` knows nothing about whether the ticket
was also expired. Predict the answer, then run: Bell asks with a revoked, expired,
unknown-type, double-booked ticket.

In [ ]:
wreck = VIEW.model_copy(update={"revoked": True, "service_type": 9})
verdict = predicate(wreck, owner=fx.ADA, requester=fx.BELL, now=fx.WINDOW.end + 999, active_ids={7})
print("fails who + expired + revoked + scope + conflict at once →", verdict.value)
assert verdict == ErrorCode.E_NOT_OWNER   # first check, first answer

**✏️ Your turn 3.1 — scope is in the eye of the controller**

Make the predicate answer `E_SCOPE` for Ada's *perfectly good* bandwidth ticket —
without touching the view, the requester, or the time. Hint: the sixth parameter.
(This is exactly how 05's stub scoped itself: it passes `known_service_types=(0,)`.)
Success: the un-commented assert passes.

In [ ]:
telemetry_only = (1,)   # a controller that can only translate telemetry

# TODO: pass known_service_types=telemetry_only so the bandwidth ticket is out of scope
verdict = predicate(VIEW, fx.ADA, fx.ADA, IN_WINDOW, set())
print("verdict →", verdict)

# un-comment once done:
# assert verdict == ErrorCode.E_SCOPE

<details><summary>✅ Solution 3.1 — peek only after trying</summary>

```python
verdict = predicate(VIEW, fx.ADA, fx.ADA, IN_WINDOW, set(), known_service_types=telemetry_only)
assert verdict == ErrorCode.E_SCOPE
```

"Has a translator" is caller-dependent by definition — the same ticket is in scope for
one controller and out of scope for another; the default is `domain.py`'s
`KNOWN_SERVICE_TYPES = (0, 1)`.

</details>

## 4. The state machine is data you can print

Once the bouncer says yes, a **session** exists, and a session has a life: requested →
authorized → active → torn down (or failed along the way). A **state machine** is
exactly that discipline: a fixed set of states plus the *legal* moves between them — you
met the idea with 05's stub sessions. The repo stores it not as `if`-chains but as a
**dict with tuple keys** `(state, event) → next state`, which buys three things: you can
*print* the whole diagram, a test can check it's complete, and there is exactly one
place a new arrow could ever be added.

In [ ]:
for (state, event), nxt in TRANSITIONS.items():
    print(f"{state.value:10} --{event:16}→ {nxt.value}")
assert len(TRANSITIONS) == 5   # five arrows; anything else is illegal or absorbed

Three kinds of move, three behaviors — and the third is a rule with a number.
A drawn arrow moves the session. A move the diagram *doesn't* draw raises
`IllegalTransition` — a **programming error** in the caller, never a user-facing denial
(users get `ErrorCode`s; impossible moves get exceptions). And terminal states **absorb**
`teardown` silently: tearing down what is already down returns the same state, no error.
That absorption is **rule 8** — *teardown is idempotent* (idempotent = doing it twice
equals doing it once) — and §9 shows why the whole system leans on it.

In [ ]:
step = transition(SessionState.REQUESTED, "authorize")
print("requested --authorize→", step.value, "  ✓ a drawn arrow")
assert step == SessionState.AUTHORIZED

again = transition(SessionState.TORN_DOWN, "teardown")
print("torn_down --teardown →", again.value, "  ✓ absorbed (rule 8: already down = success)")
assert again == SessionState.TORN_DOWN

try:
    transition(SessionState.REQUESTED, "provision_ok")   # tried to skip authorization!
    raise AssertionError("should have raised")
except IllegalTransition as err:
    print("requested --provision_ok→ ✗ IllegalTransition:", err)

**✏️ Your turn 4.1 — absorbed or illegal? (predict, then check)**

Two moves the table above doesn't list: `("failed", "teardown")` and
`("active", "authorize")`. One absorbs, one raises. Write your predictions, run, then
un-comment the assert. Success: you can explain *why* the two are treated differently.

In [ ]:
# TODO: predict each — "absorbs" or "raises"
prediction_failed_teardown = "?"
prediction_active_authorize = "?"

print("failed --teardown →", transition(SessionState.FAILED, "teardown").value)
try:
    transition(SessionState.ACTIVE, "authorize")
except IllegalTransition as err:
    print("active --authorize → IllegalTransition:", err)

# un-comment once done:
# assert transition(SessionState.FAILED, "teardown") == SessionState.FAILED

<details><summary>✅ Solution 4.1 — peek only after trying</summary>

```python
prediction_failed_teardown = "absorbs"    # FAILED is terminal — teardown converges, rule 8
prediction_active_authorize = "raises"    # no drawn arrow: re-authorizing a live session is a bug
assert transition(SessionState.FAILED, "teardown") == SessionState.FAILED
```

Teardown must always be safe to call (crash recovery, duplicate events); every *other*
undrawn move is a caller bug worth crashing on loudly.

</details>

## 5. Rule 4, proven with your own eyes

The claims in §2 — no I/O, no clock — are checkable, not vibes. Python can hand you the
**AST** (abstract syntax tree: the parsed structure of source code, before it runs) of
any module, and the import statements are right there in it. The repo's test suite runs
exactly this scan (`controller/tests/test_domain.py`); run it yourself:

In [ ]:
import ast


def top_level_imports(module) -> set[str]:
    """Every top-level package the module's source imports, read from its AST."""
    tree = ast.parse(inspect.getsource(module))
    found = set()
    for node in ast.walk(tree):
        if isinstance(node, ast.Import):
            found |= {alias.name.split(".")[0] for alias in node.names}
        elif isinstance(node, ast.ImportFrom) and node.module:
            found.add(node.module.split(".")[0])
    return found


print("domain.py imports:", sorted(top_level_imports(domain)))
assert top_level_imports(domain) <= {"__future__", "a2a_interfaces"}
print("no web3, no pygnmi, no HTTP, no filesystem — the judgment is portable ✓")

**✏️ Your turn 5.1 — scan the neighbors**

Run the same scan on `controller.translators` and `controller.auth`. Before running:
predict which module `auth` pulls in that `domain` and `translators` don't (hint: §6's
job needs cryptography). Success: your predicted name shows up, and *only* stdlib +
crypto + interfaces — still no chain, no network, no clock.

In [ ]:
from controller import auth, translators

print("translators.py →", sorted(top_level_imports(translators)))

# TODO: scan `auth` too (write your prediction as a comment first)
scanned = set()
print("auth.py        →", sorted(scanned))

# un-comment once done:
# assert "eth_account" in scanned and "secrets" in scanned

<details><summary>✅ Solution 5.1 — peek only after trying</summary>

```python
scanned = top_level_imports(auth)
assert "eth_account" in scanned and "secrets" in scanned
```

`auth` may use *crypto* (verification only — rule 2) but still no I/O: every fact it
judges — the owner, the time — arrives as an argument, same discipline as the predicate.

</details>

## 6. Challenge–response: prove you hold the key, show nobody the key

The predicate's first check compares `requester` to `owner` — but over a network,
*anyone can type Ada's address into a request*. Owning ticket #7 is on-chain fact;
**being the one asking right now** is not. The gap is closed by a **challenge–response**
dance:

1. The controller issues a **nonce** — a *number used once* — with a chain-time expiry.
   05's stub handed out guessable `nonce-0`, `nonce-1`; the real store draws 16 random
   bytes from a CSPRNG (a cryptographically secure random generator — unguessable).
2. The owner signs the agreed string
   `a2a-activate|{controller_id}|{nonce}|{entitlement_id}|{expires_at}` with plain
   EIP-191 `personal_sign` — the message signing you met in
   `07_chainmcp_the_signing_adapter.ipynb`. No transaction, no gas, nothing on chain.
3. The controller **recovers** the signer's address from the signature and compares it
   to `ownerOf(id)`. The key itself never travels and the controller never signs
   anything (rule 2) — it only verifies, in pure Python.

Read the heart of it — `issue` and `verify`:

In [ ]:
from controller.auth import NONCE_TTL_S, AuthStore, proof_message

print(inspect.getsource(AuthStore.issue))
print(inspect.getsource(AuthStore.verify))

The single most load-bearing line is the first one of `verify`:
`self._open.pop(nonce, None)` — the nonce **burns on ANY verification attempt**,
success or failure, before anything else is checked. Hold that thought; you'll defeat
two attacks with it below.

Run the dance with a real throwaway key (keys stay inside this cell — rule 2 holds even
in notebooks). What to look for: the challenge's `expires_at` is `now + 300` *chain*
seconds, the verdict is `None`, and afterwards the ledger `_open` (internal — peeking to
learn) is empty: the nonce burned.

In [ ]:
from eth_account import Account
from eth_account.messages import encode_defunct

ada_key = Account.create("course-ada")     # a throwaway 'Ada' for this kernel
store = AuthStore("bw-ctrl-1")
NOW = IN_WINDOW

ch = store.issue(7, now=NOW)
print("challenge →", ch)
assert ch.expires_at == NOW + NONCE_TTL_S              # five minutes of CHAIN time

message = proof_message(store.controller_id, ch.nonce, 7, ch.expires_at)
print("she signs →", message)
signature = "0x" + ada_key.sign_message(encode_defunct(text=message)).signature.hex()

verdict = store.verify(7, ch.nonce, signature, owner=ada_key.address, now=NOW + 5)
print("verify    →", verdict, " ← None: the proof binds ✓")
assert verdict is None
print("the ledger after verify:", store._open, " ← the nonce burned")
assert store._open == {}

Now the attack this design exists for. A **replay attack** is the network's oldest
trick: an eavesdropper (or a proxy log, or your own re-run cell) captures a *valid*
proof and submits it again. The signature is still genuine — Ada really signed it — so
signature checking alone would wave it through. The burned nonce is what kills it:

In [ ]:
replay = store.verify(7, ch.nonce, signature, owner=ada_key.address, now=NOW + 6)
print("the exact same valid proof, again →", replay.value, " ✗")
assert replay == ErrorCode.E_NONCE_REUSED
print("replay attack: defeated — a proof is single-use, however genuine ✓")

Three more ways a proof dies, each with its own honest error. Watch the thief case
closely: signing with the *wrong* key doesn't produce an error — recovery **succeeds**
and yields the thief's own address, which simply isn't Ada's. And note what happens when
the rightful owner retries the same nonce after the thief's attempt: the failed attempt
already burned it. That is deliberate — if failures didn't burn, a thief could probe one
challenge forever, using the controller as an **oracle** (a yes/no answering machine
that leaks information attempt by attempt).

In [ ]:
thief_key = Account.create("course-thief")
ch2 = store.issue(7, now=NOW)
msg2 = proof_message(store.controller_id, ch2.nonce, 7, ch2.expires_at)
thief_sig = "0x" + thief_key.sign_message(encode_defunct(text=msg2)).signature.hex()

theft = store.verify(7, ch2.nonce, thief_sig, owner=ada_key.address, now=NOW)
print("thief signs Ada's challenge →", theft.value, " ← recovered the thief's address, not Ada's")
assert theft == ErrorCode.E_NOT_OWNER

ada_sig2 = "0x" + ada_key.sign_message(encode_defunct(text=msg2)).signature.hex()
retry = store.verify(7, ch2.nonce, ada_sig2, owner=ada_key.address, now=NOW)
print("rightful Ada retries that nonce →", retry.value, " ← the failed attempt burned it (no oracle)")
assert retry == ErrorCode.E_NONCE_REUSED

Staleness and cross-binding both answer `E_BAD_PROOF` — "your proof itself is wrong,
ask for a fresh challenge won't help you impersonate anyone". Staleness is judged
against **chain time** (ADR-004 again — the `now` argument, never a wall clock), and a
nonce is bound to exactly ONE ticket at issue time:

In [ ]:
ch3 = store.issue(7, now=NOW)
msg3 = proof_message(store.controller_id, ch3.nonce, 7, ch3.expires_at)
sig3 = "0x" + ada_key.sign_message(encode_defunct(text=msg3)).signature.hex()
stale = store.verify(7, ch3.nonce, sig3, owner=ada_key.address, now=NOW + NONCE_TTL_S + 1)
print("one chain-second past expiry  →", stale.value)
assert stale == ErrorCode.E_BAD_PROOF

ch4 = store.issue(7, now=NOW)                          # issued for ticket #7...
msg4 = proof_message(store.controller_id, ch4.nonce, 8, ch4.expires_at)
sig4 = "0x" + ada_key.sign_message(encode_defunct(text=msg4)).signature.hex()
crossed = store.verify(8, ch4.nonce, sig4, owner=ada_key.address, now=NOW)   # ...spent on #8
print("nonce for #7 spent on #8      →", crossed.value, " ← a nonce binds to ONE ticket")
assert crossed == ErrorCode.E_BAD_PROOF

**✏️ Your turn 6.1 — tamper with one character**

Build the proof message with the WRONG controller id (`"bw-ctrl-2"`), sign it with
Ada's *correct* key, and verify. Predict the error first — it is **not** `E_BAD_PROOF`.
Success: the error name tells you the store rebuilt the message with its *own* id and
recovered a stranger.

In [ ]:
prediction = "?"   # TODO: which ErrorCode? (hint: recovery *succeeds* — over a different message)

ch5 = store.issue(7, now=NOW)
# TODO: change the controller_id below to "bw-ctrl-2"
tampered_msg = proof_message("bw-ctrl-1", ch5.nonce, 7, ch5.expires_at)
tampered_sig = "0x" + ada_key.sign_message(encode_defunct(text=tampered_msg)).signature.hex()

verdict = store.verify(7, ch5.nonce, tampered_sig, owner=ada_key.address, now=NOW)
print("verdict →", verdict)

# un-comment once done:
# assert verdict == ErrorCode.E_NOT_OWNER

<details><summary>✅ Solution 6.1 — peek only after trying</summary>

```python
tampered_msg = proof_message("bw-ctrl-2", ch5.nonce, 7, ch5.expires_at)
tampered_sig = "0x" + ada_key.sign_message(encode_defunct(text=tampered_msg)).signature.hex()
verdict = store.verify(7, ch5.nonce, tampered_sig, owner=ada_key.address, now=NOW)
assert verdict == ErrorCode.E_NOT_OWNER
```

`verify` rebuilds the message from its OWN `controller_id`; recovering a signature over
a *different* message yields some stranger's address — so any drift in the signed string
looks like theft (`E_NOT_OWNER`), not garbage. Beginners hit this constantly: a wrong
field doesn't error, it impersonates nobody.

</details>

## 7. Translators: the ticket's terms → the exact calls, as data

The bouncer said yes — now *what exactly* should happen? The ticket says **what** Ada
bought (50 Mbps, `qos_class` 1, an opaque 32-byte `resource_id`); the hands from the
next chapter need **where and how** (`srl1`, `ethernet-1/1` → `ethernet-1/2`).
`translate()` joins the two — and, staying pure, it returns *intended calls as data*
(`ProvisionerCall`, a frozen dataclass from 01: method name + kwargs), not actions.
Someone else applies them. Read it:

In [ ]:
from controller.resource_map import load_resource_map
from controller.translators import UnmappedResource, translate

print(inspect.getsource(translate))

Note the last line: an unknown `service_type` *here* is an `AssertionError`, not an
`ErrorCode` — the predicate already gatekept scope (§2), so reaching that branch means a
bug in the controller, and bugs crash loudly (same philosophy as `IllegalTransition`).

Run it on ticket #7 and look at three things: the method **name** is exactly a method of
the `NetworkProvisioner` port from 03 (the wiring will call it via `getattr`), the
kwargs carry the canonical 50 Mbps, and the record is immutable:

In [ ]:
from dataclasses import FrozenInstanceError

from a2a_interfaces import NetworkProvisioner

rmap = load_resource_map()
calls = translate("ent7-a1", VIEW, rmap)
call = calls[0]
print("method :", call.method)
for k, v in call.kwargs.items():
    print(f"  {k:12} = {v}")
assert call.method == "apply_bandwidth" and call.kwargs["capacity_bps"] == 50_000_000
assert callable(getattr(NetworkProvisioner, call.method))   # the port really has this method ✓

try:
    call.method = "apply_qos"
    raise AssertionError("should have raised")
except FrozenInstanceError:
    print("✗ FrozenInstanceError — calls are immutable records, not editable plans")

How do you *test* output like this? Asserting each field individually re-states the code
and misses drift in fields you forgot. The repo instead uses a **golden file**: the
expected output, serialized to JSON, **reviewed by a human eye once** — that review is
the actual verification — then checked byte-for-byte forever after. Any future drift
fails as a readable JSON diff. The golden lives at
`controller/tests/goldens/bandwidth_ticket7.json`; pin the live output against it right
now (the `as_jsonable` helper is `test_translators.py`'s trick — pydantic values like
`ResolvedPath` don't equal plain dicts, so we `model_dump()` them first):

In [ ]:
import json
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().resolve().parents] if (p / ".git").exists())
golden = json.loads((ROOT / "controller/tests/goldens/bandwidth_ticket7.json").read_text())


def as_jsonable(calls_):
    """Golden form: dataclass → dict, pydantic values → plain data."""
    return [
        {"method": c.method,
         "kwargs": {k: (v.model_dump() if hasattr(v, "model_dump") else v)
                    for k, v in c.kwargs.items()}}
        for c in calls_
    ]


print(json.dumps(golden[0]["kwargs"], indent=2))
assert as_jsonable(calls) == golden
print("\ntranslate() output == the eye-reviewed golden ✓  (pinned live, not just in CI)")

**✏️ Your turn 7.1 — the telemetry twin**

Ticket #8 (the telemetry product from 04's fixtures) has its own golden. Translate
`fx.TELEMETRY_ENTITLEMENT_VIEW` with session id `"ent8-a1"` and pin it against
`telemetry_ticket8.json`. Success: the assert passes and the method is
`apply_telemetry`.

In [ ]:
telemetry_golden = json.loads((ROOT / "controller/tests/goldens/telemetry_ticket8.json").read_text())
print("golden expects method:", telemetry_golden[0]["method"])

# TODO: translate fx.TELEMETRY_ENTITLEMENT_VIEW with session_id "ent8-a1" and rmap
tele_calls = []

# un-comment once done:
# assert as_jsonable(tele_calls) == telemetry_golden

<details><summary>✅ Solution 7.1 — peek only after trying</summary>

```python
tele_calls = translate("ent8-a1", fx.TELEMETRY_ENTITLEMENT_VIEW, rmap)
assert as_jsonable(tele_calls) == telemetry_golden
```

Same map, second entry: ticket #8's `resource_id` resolves to a `ResolvedNode` and the
`service_type == 1` branch dispatches to `_telemetry` — one private helper per product.

</details>

## 8. The resource map: the ONE place ids meet devices (ADR-005)

Where did `srl1` come from? On chain, a resource is an opaque 32-byte id — the chain
must know *nothing* about Bell's topology, and `netctl` must know nothing about tickets.
ADR-005 pins the join to exactly one file: `resource_map.yaml`, sitting next to the
controller's code. If the topology changes, one file changes. Read it with your own
eyes — comments and all — then look up ticket #7 (careful: the loaded keys are **raw
bytes**, not `"0x…"` strings):

In [ ]:
from controller.resource_map import DEFAULT_MAP

print(DEFAULT_MAP.read_text())
print("loaded:", [("0x…" + k.hex()[-2:], type(v).__name__) for k, v in rmap.items()])

resolved = rmap[bytes.fromhex(f"{7:064x}")]      # raw-bytes key, not a hex string
print("ticket #7 resolves to:", resolved)
assert resolved.device == "srl1" and resolved.ingress_if == "ethernet-1/1"

Two failure modes, two very different answers. A ticket whose id is *not in the map* is
a **valid ticket at the wrong venue** — `translate` raises `UnmappedResource`, which §9's
service surfaces as the denial `E_SCOPE`. But a *malformed map file* is a config error,
and the loader is deliberately strict: it crashes **at startup**, because a controller
that discovers its map is broken mid-provision is a much worse story.

In [ ]:
import tempfile

stranger = VIEW.model_copy(update={"resource_id": b"\x99" * 32})
try:
    translate("s", stranger, rmap)
    raise AssertionError("should have raised")
except UnmappedResource as err:
    print("✗ UnmappedResource:", err, " ← valid ticket, wrong venue (service answers E_SCOPE)")

bad = Path(tempfile.mkdtemp()) / "map.yaml"
bad.write_text('"0x07":\n  kind: teleporter\n  device: srl1\n')
try:
    load_resource_map(bad)
    raise AssertionError("should have raised")
except ValueError as err:
    print("✗ ValueError:", err)
    print("   ← config errors crash at startup, never mid-provision")

**✏️ Your turn 8.1 — the empty map (predict, then check)**

Write an *empty* YAML file and load it. Predict first: crash, or a legal value — and if
legal, which one? Success: your prediction matches and you can point at the two
characters of `resource_map.py` that decide it.

In [ ]:
prediction = "?"   # TODO: "crash" or the value you expect

empty = Path(tempfile.mkdtemp()) / "empty.yaml"
empty.write_text("")

# TODO: load it with load_resource_map(empty) and print the result
result = "not loaded yet"
print("empty map →", result)

# un-comment once done:
# assert load_resource_map(empty) == {}

<details><summary>✅ Solution 8.1 — peek only after trying</summary>

```python
result = load_resource_map(empty)
assert result == {}
```

`yaml.safe_load("")` returns `None`, and the loader's `or {}` turns that into a legal
empty map — a controller that manages nothing yet is a valid controller.

</details>

## 9. `ControllerService`: the hexagon, assembled

Every piece so far is a loose organ: a predicate, a nonce store, translators, a map.
`ControllerService` is the body — and its constructor is the payoff of
`03_protocols_and_ports.ipynb`. It takes `reader: EntitlementReader` and
`provisioner: NetworkProvisioner` — the two **ports** — plus the auth store and the
map. It never asks *what* its dependencies are, only that they fit the Protocols. That
is **dependency injection**: the same class runs on cardboard fakes here and on a real
chain + real router in `11_worlds_and_profiles.ipynb`, without changing a line.

Build its world exactly the way the repo's own test fixture does (`test_api.py`): 05's
`FakeClock`/`FakeChain`, the `ScriptedProvider` quoting the canonical offer, and
`MockProvisioner` — hands that only *record* what a router would have been told
(chapter 09 opens them up). The rhythm matters: mint at 13:32, then advance chain time
*into* the window, or the predicate will greet you with `E_NOT_STARTED`.

In [ ]:
from a2a_interfaces import EntitlementReader
from controller.service import ControllerService, Denied
from e2e.skeleton.fakes import FakeChain, FakeClock
from e2e.skeleton.scripted_agents import ScriptedProvider
from netctl.mock import MockProvisioner


def fresh_world():
    """test_api.py's fixture: a seeded stage at 14:02 with ticket #7 minted.
    Returns its own throwaway owner key — proofs must be signed by #7's owner."""
    key = Account.create("course-world")
    clock = FakeClock(fx.WINDOW.start - 1680)                     # 13:32, before the window
    chain = FakeChain(clock, balances={key.address: 10**20}, next_id=fx.TICKET_ID)
    chain.fulfill(ScriptedProvider().quote(fx.BANDWIDTH_NEED), buyer=key.address)
    clock.advance(1800)                                           # 14:02, inside it
    net = MockProvisioner()
    svc = ControllerService(chain, net, AuthStore("bw-ctrl-1"), load_resource_map())
    return clock, chain, net, svc, key


clock, chain, net, svc, ada = fresh_world()
print("chain time:", hhmm(clock.now()), "| owner of #7:", chain.owner_of(7)[:10] + "…")
print("init signature:", inspect.signature(ControllerService.__init__))
assert isinstance(chain, EntitlementReader) and isinstance(net, NetworkProvisioner)
print("FakeChain fits EntitlementReader ✓   MockProvisioner fits NetworkProvisioner ✓  (rule 7)")

`activate` is the money method — every section of this chapter appears in it, in order:
verify the proof (§6) → run the predicate (§2) → check the requested action kind against
the ticket → translate (§7, `UnmappedResource` becoming `E_SCOPE` from §8) → apply each
call with `getattr(self._net, call.method)(**call.kwargs)` — data becoming invocation —
and if the hands fail mid-way, **sweep** the half-applied config before denying. Read it
top to bottom:

In [ ]:
print(inspect.getsource(ControllerService.activate))

Drive it directly (no HTTP yet). The helper below plays the *consumer's* side of §6 —
in production chainmcp signs this exact template with Ada's real key (07). Look for:
state `active`, and the mock hands holding exactly the golden call's kwargs.

In [ ]:
def sign_proof(challenge, entitlement_id, key):
    """The consumer's half of the dance: sign the agreed template with the owner key."""
    message = proof_message(challenge.controller_id, challenge.nonce,
                            entitlement_id, challenge.expires_at)
    return "0x" + key.sign_message(encode_defunct(text=message)).signature.hex()


ch = svc.challenge(7)
info = svc.activate(7, "bandwidth", ch.nonce, sign_proof(ch, 7, ada))
print("state           :", info.state.value)
print("session_id      :", info.session_id)
print("the hands heard :", net.applied[info.session_id])
assert info.state == SessionState.ACTIVE
assert net.applied[info.session_id]["capacity_bps"] == 50_000_000

At this layer a refusal is a raised exception — `Denied`, carrying the `ErrorCode` (the
HTTP layer will turn it into a status in §10). Trigger one: activate #7 *again* with a
perfectly fresh challenge and a perfectly good signature. The predicate's `active_ids`
guard — built from live sessions with a set comprehension — refuses the double-booking:

In [ ]:
ch2 = svc.challenge(7)
try:
    svc.activate(7, "bandwidth", ch2.nonce, sign_proof(ch2, 7, ada))
    raise AssertionError("should have raised")
except Denied as err:
    print("second activation →", err.code.value, " ✗ no double-booking")
    assert err.code == ErrorCode.E_CONFLICT

Rule 8 at the service layer. Tearing down a session that *never existed* answers
`torn_down` — and, belt-and-braces, still tells the hands to clear any stray config by
that name (crash recovery: the controller restarted, the router might remember a
session the fresh process doesn't). Then tear the real session down twice:

In [ ]:
ghost = svc.teardown("never-existed")
print("teardown('never-existed') →", ghost.state.value, " ← rule 8: success, not error")
assert ghost.state == SessionState.TORN_DOWN and "never-existed" in net.torn_down

down = svc.teardown(info.session_id)
again = svc.teardown(info.session_id)
print("real teardown →", down.state.value, " | teardown again →", again.state.value, " ✓ idempotent")
assert down.state == again.state == SessionState.TORN_DOWN
assert info.session_id not in net.applied          # the recorded config is gone

What if the *router* says no? Sabotage the hands: a provisioner whose `apply_bandwidth`
always fails (we poke the internal `_net` — flagged — keeping everything else intact).
Watch two things: the denial is `E_NETWORK`, and the half-applied session was **swept
immediately** — the failure path calls teardown *before* denying, so partial config
never lingers on a router. This is why rule 8 matters so much: that sweep must be safe
to fire at any moment, from any state.

In [ ]:
from a2a_interfaces import ApplyResult


class RefusingProvisioner(MockProvisioner):
    """Hands that always fail — a stand-in for an unreachable router."""

    def apply_bandwidth(self, session_id, path, capacity_bps, qos_class):
        return ApplyResult(ok=False, detail="lab unreachable")


refusing = RefusingProvisioner()
svc._net = refusing                     # sabotage the hands, keep the rest of the body

ch3 = svc.challenge(7)
try:
    svc.activate(7, "bandwidth", ch3.nonce, sign_proof(ch3, 7, ada))
    raise AssertionError("should have raised")
except Denied as err:
    print("the router said no →", err.code.value)
    assert err.code == ErrorCode.E_NETWORK
print("swept immediately:", refusing.torn_down, " ← half-applied config never lingers ✓")
assert refusing.torn_down
svc._net = net                          # restore the honest hands

Finally, the two *autonomous* teardown paths — nobody sends an HTTP request; the
controller acts on its own. You watched both on the stub in 05; here is the time-driven
one on the real service: `tick()` re-reads **chain time** (ADR-004 — never the OS
clock) and ends every active session past its `expires_at`. The event-driven twin,
`handle_revoked`, stars in §12 — and the full revocation showpiece, on a real chain,
is `12_grand_finale.ipynb`.

In [ ]:
ch4 = svc.challenge(7)
live = svc.activate(7, "bandwidth", ch4.nonce, sign_proof(ch4, 7, ada))
assert live.state == SessionState.ACTIVE

clock.advance(fx.WINDOW.end - clock.now())         # a block lands at exactly 16:00
ended = svc.tick()
print("chain time:", hhmm(clock.now()), "→ tick() ended:", ended)
assert ended == [live.session_id]
assert svc.session(live.session_id).state == SessionState.TORN_DOWN

**✏️ Your turn 9.1 — the wrong action kind**

A fresh world, a fresh challenge — now ask for a **telemetry** action with the
*bandwidth* ticket #7, correctly signed. Predict the `ErrorCode` first. Success: the
assert passes, and you can say which line of `activate` (not the predicate!) caught it.

In [ ]:
wclock, wchain, wnet, wsvc, wkey = fresh_world()
challenge = wsvc.challenge(7)
verdict = "not attempted yet"

# TODO: call wsvc.activate(7, ...) asking for a "telemetry" action, signed with
#       sign_proof(challenge, 7, wkey); catch Denied as err and keep err.code in verdict

print("verdict →", verdict)

# un-comment once done:
# assert verdict == ErrorCode.E_SCOPE

<details><summary>✅ Solution 9.1 — peek only after trying</summary>

```python
try:
    wsvc.activate(7, "telemetry", challenge.nonce, sign_proof(challenge, 7, wkey))
except Denied as err:
    verdict = err.code
assert verdict == ErrorCode.E_SCOPE
```

The proof binds and the predicate passes (`service_type` 0 *is* known) — it is the next
line, `_ACTION_KINDS.get(action_kind) != view.service_type`, that refuses a telemetry
*action* on a bandwidth *ticket*: you can only ask for what you actually bought.

</details>

## 10. FastAPI from zero: the door gets an address

So far *we* called `svc.activate(...)` — but Ada's agent is a different program, likely
on a different machine. Programs talk across machines with **HTTP**, the web's
request/response protocol: a client sends a request naming a **verb** (`GET` = read
something, `POST` = do/create something) and a **path** (like `/v0/challenge`, also
called a **route** or **endpoint**); a server answers with a **status code** — `2xx`
success, `4xx` "your fault" (bad request), `5xx` "my fault" (server broke) — and, in a
**JSON API** like ours, a JSON body. A real server *listens* on a network **port** (a
numbered socket on a machine, e.g. 8000) — yet a third meaning of "port" after 03's
architecture ports; context disambiguates.

**FastAPI** is a Python web framework: you write plain functions, decorate them with
routes, and it handles the HTTP. That decorator is not magic — it is 01's
decorator-as-registration pattern, and you can build the essence in ten lines:

In [ ]:
ROUTES = {}


def route(path):
    """A decorator FACTORY (01): takes the path, returns the decorator."""
    def register(fn):
        ROUTES[path] = fn      # registration is the side effect...
        return fn              # ...the function returns untouched
    return register


@route("/hello")
def hello():
    return {"greeting": "hi"}


print("route table :", ROUTES)
print("dispatch '/hello' →", ROUTES["/hello"]())
assert ROUTES["/hello"] is hello

**✏️ Your turn 10.1 — extend the toy router**

Register a second route `"/bye"` returning `{"farewell": "bye"}`, then dispatch it.
Success: the assert passes — you've written the heart of every web framework.

In [ ]:
# TODO: use @route("/bye") to register a function bye() returning {"farewell": "bye"}

print("routes registered:", list(ROUTES))

# un-comment once done:
# assert ROUTES["/bye"]() == {"farewell": "bye"}

<details><summary>✅ Solution 10.1 — peek only after trying</summary>

```python
@route("/bye")
def bye():
    return {"farewell": "bye"}

assert ROUTES["/bye"]() == {"farewell": "bye"}
```

`@app.post("/v0/activate")` in the real file is this exact trick, industrial-strength:
registration at import time, dispatch at request time.

</details>

Now the real thing. `build_app(service)` is a **factory**: routes are nested functions
closing over the injected `service` — no globals, trivially testable. FastAPI adds what
the toy lacked: it parses and **validates** each JSON body against the pydantic request
models from 02 (`ChallengeRequest`, `ActivateRequest`, …) *before your code runs*, and
serializes whatever you return. And the file is deliberately **logic-free** — the house
rule says it plainly: *if an `if` about entitlements appears in `app.py`, it is in the
wrong file.* The only "decision" is the `ErrorCode → status` table, and it is data:

In [ ]:
from controller.app import STATUS, build_app

for error_code, status in STATUS.items():
    print(f"{status}  ←  {error_code.value}")
assert set(STATUS) == set(ErrorCode)     # the table is TOTAL over the enum — no code can fall through
print("\n404 = no such thing · 401 = your proof failed · 403 = the ticket says no · 502 = the network said no")

Read `build_app` and verify the no-logic claim mechanically: one exception handler turns
any raised `Denied` into `STATUS[code]` + a JSON error body, four routes parse-call-
return — and the words a judgment would need (`predicate`, `revoked`, `start_time`)
appear nowhere in the file:

In [ ]:
from controller import app as app_module

print(inspect.getsource(build_app))
src = inspect.getsource(app_module)
assert "predicate" not in src and "revoked" not in src and "start_time" not in src
print("no predicate, no revoked, no start_time in app.py ✓ — parsing and judging are different layers")

## 11. Drive the whole API — in-process, no server

You'd expect to need a running server now. Not in a notebook: FastAPI's `TestClient` is
a real HTTP client (httpx underneath) that speaks to the app **in-process** — real
requests, real routing, real status codes, no port, no network. Fresh world, then the
whole lifecycle over HTTP, starting with the challenge:

In [ ]:
import warnings

with warnings.catch_warnings():
    warnings.simplefilter("ignore")            # starlette nags about httpx versions — not today's lesson
    from fastapi.testclient import TestClient

hclock, hchain, hnet, hsvc, hkey = fresh_world()
api = build_app(hsvc)
client = TestClient(api)

resp = client.post("/v0/challenge", json={"entitlement_id": 7})
challenge = resp.json()
print("POST /v0/challenge →", resp.status_code, challenge)
assert resp.status_code == 200 and challenge["controller_id"] == "bw-ctrl-1"

Activate — the proof is built from the challenge's JSON exactly as §9's helper did from
the dataclass (over the wire there are only dicts). Then read the session back with the
`GET` route, whose `{session_id}` **path parameter** FastAPI binds to the function
argument. It succeeds because the world is at 14:02 — inside the window:

In [ ]:
message = proof_message(challenge["controller_id"], challenge["nonce"], 7, challenge["expires_at"])
signature = "0x" + hkey.sign_message(encode_defunct(text=message)).signature.hex()
payload = {
    "entitlement_id": 7,
    "action": {"kind": "bandwidth"},
    "proof": {"nonce": challenge["nonce"], "signature": signature},
}
resp = client.post("/v0/activate", json=payload)
body = resp.json()
print("POST /v0/activate →", resp.status_code, "state:", body["state"])
assert resp.status_code == 200 and body["state"] == "active"
assert hnet.applied[body["session_id"]]["capacity_bps"] == 50_000_000

fetched = client.get(f"/v0/sessions/{body['session_id']}")
print(f"GET  /v0/sessions/{body['session_id']} →", fetched.status_code, fetched.json())
assert fetched.json()["entitlement_id"] == 7 and fetched.json()["expires_at"] == fx.WINDOW.end

Now the failures — this is §10's STATUS table at work, turning §6 and §2's refusals into
HTTP. Replaying the exact same payload burns on the nonce → **401** (your proof failed).
A thief with his own key → **403** (the ticket says no: `E_NOT_OWNER`). A ticket that
doesn't exist → **404**. Note the helper: every *fresh* activation attempt needs a fresh
challenge — nonces are single-use, as you proved in §6.

In [ ]:
replayed = client.post("/v0/activate", json=payload)      # the exact same bytes again
print("replay         →", replayed.status_code, replayed.json())
assert (replayed.status_code, replayed.json()) == (401, {"error": "E_NONCE_REUSED"})


def http_activate(key, kind="bandwidth"):
    """test_api.py's helper: fresh challenge → sign → POST /v0/activate."""
    ch_ = client.post("/v0/challenge", json={"entitlement_id": 7}).json()
    msg_ = proof_message(ch_["controller_id"], ch_["nonce"], 7, ch_["expires_at"])
    sig_ = "0x" + key.sign_message(encode_defunct(text=msg_)).signature.hex()
    return client.post("/v0/activate", json={
        "entitlement_id": 7, "action": {"kind": kind},
        "proof": {"nonce": ch_["nonce"], "signature": sig_},
    })


thief = http_activate(Account.create("course-http-thief"))
print("thief's proof  →", thief.status_code, thief.json())
assert (thief.status_code, thief.json()) == (403, {"error": "E_NOT_OWNER"})

missing = client.post("/v0/challenge", json={"entitlement_id": 99})
print("ticket #99     →", missing.status_code, missing.json())
assert (missing.status_code, missing.json()) == (404, {"error": "E_UNKNOWN_ENTITLEMENT"})

One failure class never reaches the controller at all. Send a *shape* error —
`entitlement_id` as the word `"seven"` — and pydantic rejects it with **422**
(Unprocessable Entity) before a single line of controller code runs. This is 02's
`ValidationError` wearing an HTTP status: parsing and judging are different layers,
with different owners and different error vocabularies.

In [ ]:
bad_shape = client.post("/v0/challenge", json={"entitlement_id": "seven"})
print("entitlement_id='seven' →", bad_shape.status_code)
print("pydantic says:", bad_shape.json()["detail"][0]["msg"])
assert bad_shape.status_code == 422        # rejected before controller code ran

Teardown over HTTP — twice, because rule 8 has an HTTP face too. Same request, same
answer, no error. (Compare: a non-idempotent API would 404 or 409 on the second call,
and every crashed client that retries would break.)

In [ ]:
down = client.post("/v0/teardown", json={"session_id": body["session_id"]})
down_again = client.post("/v0/teardown", json={"session_id": body["session_id"]})
print("teardown →", down.json(), " | teardown again →", down_again.json(), " ✓ rule 8 over HTTP")
assert down.json() == down_again.json() == {"state": "torn_down"}

A free bonus of building on types: FastAPI generated the machine-readable API
description (**OpenAPI** — the standard JSON schema-of-the-API format) from the
decorators and pydantic models alone. Four routes, exactly docs/03 §3:

In [ ]:
paths = sorted(api.openapi()["paths"])
print("\n".join(paths))
assert paths == ["/v0/activate", "/v0/challenge", "/v0/sessions/{session_id}", "/v0/teardown"]

**✏️ Your turn 11.1 — expiry over HTTP (predict, then check)**

Advance this world's chain clock three hours — well past 16:00 — and try a *fresh,
correctly signed* activation with `http_activate(hkey)`. Predict the status code AND the
error string first (the STATUS table in §10 is your map). Success: the un-commented
assert passes with your predictions.

In [ ]:
prediction_status = 0        # TODO: your predicted status code
prediction_error = "?"       # TODO: your predicted error string

hclock.advance(3 * 3600)     # chain time jumps far past the window's end
resp = http_activate(hkey)
print("after the window →", resp.status_code, resp.json())

# un-comment once done:
# assert (resp.status_code, resp.json()) == (prediction_status, {"error": prediction_error})

<details><summary>✅ Solution 11.1 — peek only after trying</summary>

```python
prediction_status = 403
prediction_error = "E_EXPIRED"
assert (resp.status_code, resp.json()) == (403, {"error": "E_EXPIRED"})
```

The proof still binds (401 family doesn't fire); the *predicate* answers `E_EXPIRED`,
and STATUS maps every "the ticket says no" denial — not-owner, not-started, expired,
revoked, scope, conflict — to 403.

</details>

## 12. Who calls `tick()`? The wiring, glossed

In §9 *you* called `tick()` by hand. In production nobody does — `wiring.py` is where
the ports meet reality, and it is the **only** controller module that touches concrete
adapters. Its `build_runtime(rpc_url, provisioner, …)` composes a read-only
`ChainReader` (the controller holds **no key**, not even an unused one — rule 2) with a
real provisioner into a `ControllerService`, deferring `from chainmcp import
ChainReader` to *inside* the function so the module imports cleanly without a chain.
You'll call it against a live Anvil in `11_worlds_and_profiles.ipynb`.

But its two *drivers* run on cardboard right here. `ExpiryTimer` is ADR-004 in code: a
**daemon thread** (from 01 — daemon = dies with the process, so a forgotten timer can't
hang the notebook) that polls `tick()` every `poll_s` seconds. One piece of its source
is new — 01 gave you `Thread` and `daemon`, not `threading.Event`. An `Event` is a
thread-safe flag: `.wait(timeout)` sleeps up to `timeout` seconds but returns instantly
(with `True`) the moment another thread calls `.set()` — so the loop line
`while not self._stop.wait(self._poll_s):` is the sleep AND the stop switch in one
object. Read the source and notice what the thread does NOT do — it never computes
"when" anything expires. The OS timer only schedules *wake-ups*; chain time, re-read
inside `tick()`, makes every decision:

In [ ]:
from controller.wiring import ExpiryTimer

print(inspect.getsource(ExpiryTimer))

Run it: activate a session, start a fast-polling timer, then jump chain time to 16:00.
Within a poll or two the session dies — no HTTP call, no human. Always `stop()` a timer
you started: it calls `.set()`, the sleeping `wait` wakes at once, and the thread
exits instead of lingering a full poll:

In [ ]:
tclock, tchain, tnet, tsvc, tkey = fresh_world()
ch = tsvc.challenge(7)
live = tsvc.activate(7, "bandwidth", ch.nonce, sign_proof(ch, 7, tkey))
assert live.state == SessionState.ACTIVE

timer = ExpiryTimer(tsvc, poll_s=0.2)
timer.start()
tclock.advance(fx.WINDOW.end - tclock.now())     # 16:00 lands on the fake chain
time.sleep(0.7)                                  # give the poll a couple of wake-ups
timer.stop()                                     # ALWAYS stop the thread you started

state = tsvc.session(live.session_id).state
print("no HTTP call, no human — the timer noticed chain time:", state.value, "✓")
assert state == SessionState.TORN_DOWN

The event-driven twin needs no thread at all here: `ControllerRuntime.start_watchers`
wires the chain adapter's `watch_revoked` callback straight into
`service.handle_revoked` — and `FakeChain` notifies synchronously, so one `revoke()`
call runs the whole path. Two details worth savoring: `handle_revoked` **re-reads the
chain** before acting (never trust the event — the chain is the source of truth), and a
duplicate event lands on rule 8's absorption, so it simply converges. The revocation
showpiece — this exact path against a *real* chain, killing a session mid-window — is
`12_grand_finale.ipynb`.

In [ ]:
rclock, rchain, rnet, rsvc, rkey = fresh_world()
ch = rsvc.challenge(7)
live = rsvc.activate(7, "bandwidth", ch.nonce, sign_proof(ch, 7, rkey))

rchain.watch_revoked(rsvc.handle_revoked)      # what ControllerRuntime.start_watchers wires
rchain.revoke(7)                               # Bell pulls the ticket...
state = rsvc.session(live.session_id).state
print("revoke(7) fired the watcher → session:", state.value, "✓")
assert state == SessionState.TORN_DOWN

rsvc.handle_revoked(7)                         # ...and a duplicate event is harmless
print("duplicate event → still", rsvc.session(live.session_id).state.value, " ✓ rule 8 absorbs")

**✏️ Your turn 12.1 — prove the timer is honest about time**

Fresh world, active session, *running* `ExpiryTimer` — now sleep one full REAL second
without touching the fake clock, and predict the session state before you look. Then
stop the timer. Success: your prediction matches, and you can recite ADR-004 from
memory: *OS timers schedule wake-ups; chain time decides.*

In [ ]:
prediction = "?"   # TODO: "active" or "torn_down" — commit before running

xclock, xchain, xnet, xsvc, xkey = fresh_world()
ch = xsvc.challenge(7)
live = xsvc.activate(7, "bandwidth", ch.nonce, sign_proof(ch, 7, xkey))
timer = ExpiryTimer(xsvc, poll_s=0.2)
timer.start()
time.sleep(1.0)          # one REAL second — about five polls — but chain time never moves
timer.stop()

state = xsvc.session(live.session_id).state
print("after 1 real second of polling →", state.value)

# un-comment once done:
# assert state == SessionState.ACTIVE

<details><summary>✅ Solution 12.1 — peek only after trying</summary>

```python
prediction = "active"
assert state == SessionState.ACTIVE
```

The timer woke ~5 times and called `tick()` every time — and every `tick()` re-read
`chain_time()`, still 14:02, and did nothing. Real seconds mean nothing; only the chain's
clock ends sessions (ADR-004).

</details>

## What you learned (and where it goes)

| You did | The concept | Where it goes next |
|---|---|---|
| ran the accept case and all six deny paths | the predicate: pure function, `ErrorCode \| None`, time as an argument | agents branch on these exact strings (10); benchmarked in the evaluation (13) |
| wrecked every fact at once → `E_NOT_OWNER` | the deny **order** is a documented contract — first check answers | stable errors tests and agents can pin |
| printed `TRANSITIONS`, absorbed a re-teardown | the state machine as data; rule 8 terminal absorption | tick & revocation can fire at any moment, safely (12) |
| AST-scanned `domain.py`'s imports | rule 4: the judgment imports no I/O — it is portable and replayable | the same scan is a pinned test in `controller/tests/` |
| signed a challenge, then replayed it | nonce = number used once; burn-on-any-attempt (no oracle); chain-time expiry | chainmcp signs the same template for real (07 → 12) |
| pinned `translate()` against the golden JSON | golden files: eye-reviewed once, guarded forever | any drift = a readable diff in CI |
| read `resource_map.yaml`, hit `UnmappedResource` | ADR-005: ONE file where resourceIds meet devices | Bell sells a new resource = one new entry |
| assembled `ControllerService` on fakes, sabotaged the hands | the hexagon: injected ports; sweep half-applied config, then deny | 11 swaps in ChainReader + GnmiProvisioner unchanged |
| drove the API to 200/401/403/404/422 | FastAPI parses, the service judges, STATUS maps — three layers, no leaks | Ada's agent calls these routes (10, 12) |
| watched `ExpiryTimer` and `watch_revoked` end sessions | ADR-004: timers schedule wake-ups, chain time decides | the revocation showpiece on a real chain (12) |

## Experiments to try

Predict the outcome *before* running each:

- In `controller/src/controller/domain.py`, swap the `revoked` and `expired` checks
  (restore with `git checkout -- .` after). Predict which *named* test catches the
  reorder — then run `uv run pytest controller/` and find that none does: each
  deny-path test flips exactly ONE fact, so no test ever holds a ticket that is both
  expired and revoked — that part of the order is unpinned. The swap is still
  observable: ask the predicate about §3's `wreck` with `requester=fx.ADA` and the
  verdict flips from `E_EXPIRED` to `E_REVOKED`. A contract line no test pins is
  itself an honest finding — a wreck-style ordering test would close the gap.
- Rename `capacity_bps` → `capacityBps` inside `_bandwidth` and run the golden test —
  read the failure. It is a JSON diff a human can review; that reviewability is the
  whole point of golden files.
- Call `store.verify(7, ch.nonce, "0xzz", owner=ada_key.address, now=NOW)` with a fresh
  challenge — malformed signature bytes. Which `except` branch answers, and why is the
  deliberately broad `except Exception` justified *there* and almost nowhere else?
- Deliberate breakage: delete `ErrorCode.E_CONFLICT` from a **copy** of the STATUS dict
  and imagine the double-booking over HTTP — what happens inside the exception handler?
  (§10's totality assert is the guard; the repo keeps the table total so no denial can
  fall through to a 500.)

## Check yourself

1. Rule 1: why must the bouncer be deterministic code — and what are the only two
   places in the whole system where LLM judgment is allowed?
2. The predicate takes `now` as an argument instead of reading a clock. Who supplies
   it, from which clock, and what does that buy for testing and for disputes (ADR-004)?
3. Ada signs a proof with her correct key but for controller `"bw-ctrl-2"`. Which
   `ErrorCode` comes back, and why is it *not* `E_BAD_PROOF`?
4. Why does a FAILED verification burn the nonce too? What could an attacker do with a
   store that only burned on success?
5. Over HTTP, what is the difference between a 422 and a 403 — which layer produces
   each, and what does that split prove about `app.py`?

**Where this goes next:** `09_netctl_the_hands.ipynb` — the other side of the
`NetworkProvisioner` port: what `apply_bandwidth(...)` actually does to a router, and
what this lab honestly can and cannot enforce (ADR-006).

**Deeper dive:** the compact tour [`controller_explore.ipynb`](../controller_explore.ipynb)
· the rulebook [`docs/05-controller-spec.md`](../../../docs/05-controller-spec.md) ·
every claim here, pinned: [`controller/tests/`](../../../controller/tests/) ·
the decisions:
[`docs/adr/004-chain-time-canonical-clock.md`](../../../docs/adr/004-chain-time-canonical-clock.md)
and [`docs/adr/005-resourceid-resolution.md`](../../../docs/adr/005-resourceid-resolution.md).